[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kasparvonbeelen/diailectics/blob/main/interface.ipynb)

# Persona chat — Runaway frame annotation

Paste an article below. Claude first reads it against the schema in
[runaway_package_annotator_instructions.html](runaway_package_annotator_instructions.html)
and picks out the passages actually worth annotating (most of an article isn't about
the Runaway frame at all) — this replaces manually marking yes/no yourself. Each
selected passage gets a provisional yes/no + justification, and every persona from
[persona_test_prompts.md](persona_test_prompts.md) you select comments on it —
including its **own yes/no stance badge** — run **statelessly** at first (matches
the `[STATELESS MODE ONLY: ...]` branch of each prompt).

The article preview above just highlights the selected passages (`[1]`, `[2]`, ...).
The interactive panel below it is where the actual annotation happens: expand a
persona to read its full response, then **write a reply in the box underneath it**
— it's sent back to that same persona as a follow-up turn (with the original
passage, schema, and its own prior response still in context), and its response,
stance badge, and summary all update in place.

**Cost note:** the initial run makes `1 + (passages found × personas selected)` API
calls, run concurrently — e.g. 4 passages × 4 personas = 17 calls. Each reply you
send afterward is one more call. Tune "Max passages" and the persona checkboxes to
control cost.

**Before running:** set `ANTHROPIC_API_KEY` in your shell environment, add a `.env`
file in the project root, or drop the key in `api_key.json`'s `anthropic_api_key`
field (`tools/claude_chat.py` checks all three, in that order, and never prints
the key).

**Note on Adorno:** in `persona_test_prompts.md`, Adorno compares a *set* of outlier
cases against the schema, not a single annotation. Here it's scoped down to "does
this one passage, given its annotation, expose a tension in the schema" — see the
comment above `TASK_LINES` in `tools/claude_chat.py`.

## Running this on Google Colab

Click the badge above, or open
`https://colab.research.google.com/github/kasparvonbeelen/diailectics/blob/main/interface.ipynb`
directly. Then just run the cells top to bottom:

1. **"Colab setup"** clones this repo into `/content/diailectics` and installs
   `requirements.txt` (only runs its clone/install steps when it detects it's
   actually on Colab - it's a no-op if you're running locally instead).
2. **"Authentication"** looks for credentials in this order: an `ANTHROPIC_API_KEY`
   already in the environment, a Colab secret named `ANTHROPIC_API_KEY` (key icon
   in the left sidebar - the recommended way, since it's never stored in the
   notebook or repo), an uploaded `api_key.json` (drop it in the Colab file
   browser - either at `/content/api_key.json` or directly into `diailectics/`,
   either location is picked up automatically), or - as a last resort on Colab
   only - a prompt to paste the key directly (kept in memory for the session only).
3. Run the remaining cells as normal.


### Colab setup (no-op if not on Colab)

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/kasparvonbeelen/diailectics.git"
REPO_DIR = "/content/diailectics"

if IN_COLAB:
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
        check=True,
    )

    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Running on Colab. Cloned/updated {REPO_DIR}, installed requirements, cwd set there.")
else:
    print("Not running on Colab - using the local checkout as-is (pip install -r requirements.txt yourself if needed).")


### Authentication

In [ ]:
import os
import shutil
from getpass import getpass


def _find_and_stage_api_key_json(repo_dir):
    """tools/claude_chat.py looks for api_key.json next to itself (repo_dir).
    Colab users typically drop uploads into /content/ instead, so check there
    too and copy it into place if that's where it landed."""
    target = os.path.join(repo_dir, "api_key.json")
    if os.path.exists(target):
        return True
    for candidate_dir in ("/content", os.path.expanduser("~")):
        candidate = os.path.join(candidate_dir, "api_key.json")
        if os.path.exists(candidate):
            shutil.copy(candidate, target)
            return True
    return False


def _try_colab_secret():
    if not IN_COLAB:
        return False
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        return False
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key
        return True
    return False


if os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN"):
    print("ANTHROPIC_API_KEY (or ANTHROPIC_AUTH_TOKEN) is already set in the environment.")
elif _try_colab_secret():
    print("Loaded ANTHROPIC_API_KEY from Colab secrets (the key icon in the left sidebar).")
elif _find_and_stage_api_key_json(REPO_DIR if IN_COLAB else "."):
    print("Found api_key.json - tools/claude_chat.py will read its 'anthropic_api_key' field automatically.")
elif IN_COLAB:
    print("No credentials found yet. Options, in order of preference:")
    print("  1. Add a Colab secret named ANTHROPIC_API_KEY (key icon in the left sidebar), or")
    print("  2. Upload an api_key.json file (with an 'anthropic_api_key' field) into this session, or")
    print("  3. Paste your key below (kept only in memory for this session).")
    entered = getpass("Anthropic API key (leave blank to skip): ").strip()
    if entered:
        os.environ["ANTHROPIC_API_KEY"] = entered
        print("Key set for this session.")
    else:
        print("No key set - ChatBackend() will fail until one of the options above is provided.")
else:
    print(
        "No ANTHROPIC_API_KEY / api_key.json found locally. Set ANTHROPIC_API_KEY in your "
        "shell, add a .env file, or place api_key.json next to tools/claude_chat.py - see README.md."
    )


In [ ]:
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from tools.claude_chat import (
    ChatBackend,
    PERSONA_META,
    PERSONA_KEYS,
    extract_stance,
    one_sentence_summary,
    render_html,
    render_article_marks_html,
)


In [ ]:
# Loads persona_test_prompts.md + runaway_package_annotator_instructions.html once,
# and creates the Anthropic client (see the credential order in the module docstring).
backend = ChatBackend()
print("Personas loaded:", list(backend.personas.keys()))
print("Schema loaded:", len(backend.schema), "characters")
print("Default model:", backend.model)


In [ ]:
article_box = widgets.Textarea(
    value="",
    placeholder="Paste the full article here...",
    description="Article:",
    layout=widgets.Layout(width="100%", height="220px"),
    style={"description_width": "60px"},
)

model_box = widgets.Combobox(
    value=backend.model,
    placeholder="Model ID",
    options=["claude-opus-5", "claude-sonnet-5", "claude-haiku-4-5"],
    description="Model:",
    ensure_option=False,
    style={"description_width": "60px"},
)

max_passages_slider = widgets.IntSlider(
    value=4, min=1, max=8, step=1, description="Max passages:",
    style={"description_width": "100px"},
)

persona_checkboxes = {
    key: widgets.Checkbox(value=True, description=PERSONA_META[key]["title"])
    for key in PERSONA_KEYS
}

run_button = widgets.Button(description="Annotate article", button_style="primary")
show_raw_checkbox = widgets.Checkbox(value=False, description="Show raw selection output")
output_area = widgets.Output()


def persona_panel_title(persona_key, raw_text):
    stance = extract_stance(raw_text)
    icon = "✅" if stance else ("❌" if stance is False else "❔")
    summary = one_sentence_summary(raw_text)
    title = PERSONA_META[persona_key]["title"]
    return f"{icon} {title} — {summary}"


def build_persona_accordion(passage):
    """One Accordion, one tab per persona, each with the response + a reply box
    wired to that persona's PersonaConversation."""
    accordion = widgets.Accordion()
    children = []
    entries = list(passage["personas"].items())

    for persona_key, pdata in entries:
        title_text = PERSONA_META[persona_key]["title"]
        response_html = widgets.HTML(value=pdata["html"])
        reply_box = widgets.Textarea(
            placeholder=f"Write a reply to {title_text}...",
            layout=widgets.Layout(width="100%", height="70px"),
        )
        send_button = widgets.Button(description="Send reply", button_style="")
        status_html = widgets.HTML(value="")
        reply_label = widgets.HTML(
            f'<div style="margin:10px 0 4px;font-size:12px;color:#5A5C4F;">'
            f"<b>Your reply to {title_text}:</b></div>"
        )
        panel = widgets.VBox([
            response_html,
            reply_label,
            reply_box,
            widgets.HBox([send_button, status_html]),
        ])
        children.append(panel)
        # persona_key/conversation/response_html/reply_box/status_html are captured
        # per-iteration via these default arguments (classic loop-closure fix).
        send_button.on_click(
            lambda _btn, persona_key=persona_key, conversation=pdata["conversation"],
                   response_html=response_html, reply_box=reply_box, status_html=status_html: (
                _handle_reply(accordion, persona_key, conversation, response_html, reply_box, status_html)
            )
        )

    accordion.children = children
    for i, (persona_key, pdata) in enumerate(entries):
        accordion.set_title(i, persona_panel_title(persona_key, pdata["raw_text"]))
    return accordion


def _handle_reply(accordion, persona_key, conversation, response_html, reply_box, status_html):
    text = reply_box.value.strip()
    if not text:
        return
    status_html.value = "<i>Sending reply to Claude...</i>"
    try:
        new_raw = conversation.reply(text)
    except Exception as exc:
        status_html.value = f'<span style="color:#8B3A3A;">Error: {exc}</span>'
        return
    response_html.value = render_html(persona_key, new_raw)
    reply_box.value = ""
    # Find this persona's tab by matching its own response widget, so the title
    # (icon + summary) refreshes even if tabs get reordered or rebuilt later.
    for i, child in enumerate(accordion.children):
        if child.children[0] is response_html:
            accordion.set_title(i, persona_panel_title(persona_key, new_raw))
            break
    status_html.value = f"<i>Updated (reply {conversation.reply_count}).</i>"


def build_passage_section(idx, passage):
    badge_color = "#4A7A6B" if passage["contains_runaway"] else "#8B3A3A"
    badge_text = "Runaway: yes" if passage["contains_runaway"] else "Runaway: no"
    quote = passage["quote"]
    justification = passage.get("justification", "")
    header_html = (
        f'<div style="margin:18px 0 6px;padding-top:10px;border-top:2px solid #C9C4B4;'
        f'font-family:Georgia,\'Lora\',serif;">'
        f'<div style="font-family:\'JetBrains Mono\',monospace;font-size:11px;'
        f'text-transform:uppercase;letter-spacing:.05em;color:#5A5C4F;margin-bottom:6px;">'
        f'Passage {idx}</div>'
        f'<blockquote style="margin:0 0 8px;padding:8px 14px;border-left:3px solid #C9C4B4;'
        f'background:#fff;border-radius:4px;font-style:italic;">{quote}</blockquote>'
        f'<span style="display:inline-block;font-size:11px;font-family:\'JetBrains Mono\',monospace;'
        f'text-transform:uppercase;letter-spacing:.05em;color:#fff;background:{badge_color};'
        f'padding:2px 8px;border-radius:10px;">{badge_text}</span>'
        f'<div style="color:#5A5C4F;font-size:12.5px;margin:6px 0 0;">{justification}</div>'
        f'</div>'
    )
    return widgets.VBox([widgets.HTML(header_html), build_persona_accordion(passage)])


def on_run_clicked(_):
    output_area.clear_output()
    with output_area:
        if not article_box.value.strip():
            print("Paste an article first.")
            return
        selected_personas = [key for key, box in persona_checkboxes.items() if box.value]
        if not selected_personas:
            print("Select at least one persona.")
            return
        n_calls = 1 + max_passages_slider.value * len(selected_personas)
        print(f"Calling Claude ({model_box.value})... up to {n_calls} API calls, running concurrently.")
        result = backend.annotate_article(
            article_box.value.strip(),
            personas=selected_personas,
            max_passages=max_passages_slider.value,
            model=model_box.value,
        )
        output_area.clear_output()
        n_found = len(result["passages"])
        print(f"Found {n_found} passage(s). Expand a persona below to read it in full and reply.")
        display(HTML(render_article_marks_html(article_box.value.strip(), result["passages"])))
        if show_raw_checkbox.value:
            print("\n--- raw passage-selection output ---\n")
            print(result["selection_raw"])
        for idx, passage in enumerate(result["passages"], start=1):
            display(build_passage_section(idx, passage))


run_button.on_click(on_run_clicked)

display(widgets.VBox([
    article_box,
    model_box,
    max_passages_slider,
    widgets.HTML("<b>Personas:</b>"),
    widgets.HBox(list(persona_checkboxes.values())),
    widgets.HBox([run_button, show_raw_checkbox]),
    output_area,
]))


## Calling it without the widgets

```python
# Whole article - Claude selects the passages itself
result = backend.annotate_article(
    article_text,
    personas=["devils_advocate", "hegel"],  # default: all four
    max_passages=4,
    model="claude-opus-5",                  # optional override
)
display(HTML(result["html"]))  # static hover-popover version, with stance badges

# Every persona output carries its own stance and a conversation you can reply to:
pdata = result["passages"][0]["personas"]["hegel"]
print(pdata["raw_text"])          # includes a <stance>yes|no</stance> tag
updated_raw = pdata["conversation"].reply("I don't think this reading holds up because ...")
display(HTML(render_html("hegel", updated_raw)))

# Single passage, single persona, with a yes/no annotation you already have
result = backend.run(
    text="Le réacteur est hors de contrôle...",
    persona="devils_advocate",
    contains_runaway=True,
    model="claude-sonnet-5",                # optional override
)
display(HTML(result["html"]))
result["conversation"].reply("...")         # this one supports replies too
```